In [1]:
import json
from dataclasses import dataclass, astuple
from pathlib import Path
from typing import Any, TypedDict, Literal

from drehmal_mc_name_overrides import DREHMAL_MC_NAME_OVERRIDES

In [2]:
MC_ITEM_JSON_FILE = Path("../tools/items_1_20_2.json")
MC_POTION_JSON_FILE = Path("../tools/potions.json")
DIR_DATA = Path("../data_raw/")
print(DIR_DATA.resolve())
AVAILABLE_DIMENSIONS = ["overworld", "end", "lodahr", "space", "true_end"]

C:\Users\Zachary\Coding\VSCode\DrehmalMap\data_raw


In [3]:
POTION_PREFIX = {
    "tipped_arrow": "Arrow of ",
    "splash_potion": "Splash Potion of ",
    "lingering_potion": "Lingering Potion of ",
    "potion": "Potion of "
}

NO_EFFECT_POTIONS = {"minecraft:empty", "minecraft:water", "minecraft:mundane",
                     "minecraft:thick", "minecraft:awkward"}

BLANK_SIGN_DATA = ["", "", "", ""]
BLANK_SIGN_FRONT_DATA = [" ", "", "", ""]  # There is a space in all the blank signs front_data.

In [4]:
# use parse_item(..., has_slots=False)
ENTITY_WITH_ITEMS = {"minecraft:chest_minecart", "minecraft:hopper_minecart",
                     "minecraft:chest_boat", "minecraft:chiseled_bookshelf"}
ENTITY_WITH_SINGLE_ITEM = {"minecraft:item_frame", "minecraft:glow_item_frame"}

# Don't want junk items, projectiles, marker, paintings, xp orbs
# see https://minecraft.wiki/w/Java_Edition_data_values#Entities
SKIP_ENTITIES = {"minecraft:item", "minecraft:painting", "minecraft:marker", "minecraft:area_effect_cloud", "minecraft:leash_knot",
                 "minecraft:falling_block", "minecraft:fishing_bobber", "minecraft:firework_rocket", "minecraft:potion",
                 "minecraft:tnt", "minecraft:lightning_bolt", "minecraft:llama_spit", "minecraft:fireball", "minecraft:experience_bottle",
                 "minecraft:small_fireball", "minecraft:wither_skull", "minecraft:dragon_fireball", "minecraft:egg",
                 "minecraft:ender_pearl", "minecraft:evoker_fangs", "minecraft:eye_of_end", "minecraft:experience_orb",
                 "minecraft:player", "minecraft:arrow", "minecraft:spectral_arrow", "minecraft:wind_charge",
                 "minecraft:breeze_wind_charge", "minecraft:shulker_bullet", "minecraft:snowball", "minecraft:trident",}
# 1.19 introduces `interaction` and `block/item/text_display`
# these are not in Drehmal 2.2.2 so its ok

In [5]:
def convert_json_to_dict(file_path: Path) -> dict[str, str]:
    with open(file_path, 'r') as f:
        data = json.load(f)
    
    result_dict = {item['name']: item['displayName'] for item in data}
    
    return result_dict

# Create name dicts
# overwrite with Drehmal names
MC_NAME_TO_DISPLAY = convert_json_to_dict(MC_ITEM_JSON_FILE) | DREHMAL_MC_NAME_OVERRIDES
MC_POTION_NAME_TO_DISPLAY = convert_json_to_dict(MC_POTION_JSON_FILE)
print("Items", len(MC_NAME_TO_DISPLAY))
print("Potions", len(MC_POTION_NAME_TO_DISPLAY))

Items 1294
Potions 47


In [6]:
def read_drehmal_json_data(dimension: str, group_name: Literal["block_entities", "entities"]) -> list[dict]:
    file_path = DIR_DATA / f"{dimension}_{group_name}.json"
    if not file_path.exists():
        raise FileNotFoundError(f"No file found at {file_path}")
    
    with open(file_path, 'r', encoding="utf8") as f:
        data = json.load(f)
    return data

In [7]:
# from v2.2.2 in MC 1.20.1
test_json = [
    # Runic Catalyst
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:command_block",
            "Slot": 13,
            "tag": {
            "CustomModelData": 1000000,
            "display": {
                "Lore": [
                "{\"text\":\"A small, magical orb valued by\"}",
                "{\"text\":\"traders and arcanists. They have\"}",
                "{\"text\":\"several applications in both\"}",
                "{\"text\":\"magical creations and technology.\"}"
                ],
                "Name": "{\"text\":\"Runic Catalyst\",\"color\":\"aqua\",\"italic\":false}"
            },
            "RunicCatalyst": 1
            }
        }
        ],
        "keepPacked": 0,
        "x": -192,
        "y": 73,
        "z": -347
    },
    # Same item in multiple slots
    {
        "keepPacked": 0,
        "x": -361,
        "y": 108,
        "z": 769,
        "Items": [
            {
                "Slot": 8,
                "id": "minecraft:bread",
                "Count": 1
            },
            {
                "Slot": 14,
                "id": "minecraft:paper",
                "Count": 1
            },
            {
                "Slot": 18,
                "id": "minecraft:bread",
                "Count": 1
            },
            {
                "Slot": 19,
                "id": "minecraft:paper",
                "Count": 3
            },
            {
                "Slot": 26,
                "id": "minecraft:compass",
                "Count": 1
            }
        ],
        "id": "minecraft:chest"
    },
    # Standard item with custom display name/enchantment
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:lapis_lazuli",
            "Slot": 0
        },
        {
            "Count": 1,
            "id": "minecraft:iron_nugget",
            "Slot": 1,
            "tag": {
            "display": {
                "Name": "{\"text\":\"\\\"Friendliness Pellets\\\"\",\"italic\":false}"
            },
            "Enchantments": [
                {
                "id": "minecraft:protection",
                "lvl": 1
                }
            ],
            "HideFlags": 1
            }
        },
        {
            "Count": 2,
            "id": "minecraft:iron_ingot",
            "Slot": 3
        },
        {
            "Count": 1,
            "id": "minecraft:iron_nugget",
            "Slot": 4,
            "tag": {
            "display": {
                "Name": "{\"text\":\"\\\"Friendliness Pellets\\\"\",\"italic\":false}"
            },
            "Enchantments": [
                {
                "id": "minecraft:protection",
                "lvl": 1
                }
            ],
            "HideFlags": 1
            }
        },
        ],
        "keepPacked": 0,
        "x": -504,
        "y": 71,
        "z": 946
    },
    # Lectern with book
    {
        "Book": {
        "Count": 1,
        "id": "minecraft:writable_book",
        "tag": {
            "display": {
            "Name": "{\"text\":\"Diary of a Concerned Mother\"}"
            },
            "pages": [
            "Those damn Mihkmari are spreading again. We've been avoiding the old guildhouse for a while, but nowadays we can't even scavenge in the industrial zone to the west without running into a patrol. And going to the central island would be tantamount to suicide at this point. It's ",
            "simply swarming with the gray-skinned freaks. I told my husband we should move out ages ago, but he said the loot was too good to pass up. Lotta good a bunch of broken tech will do us when those uncivilized barbarians start battering down our doors. Virtuo, be with us, I beg of you."
            ],
            "RepairCost": 0
        }
        },
        "id": "minecraft:lectern",
        "keepPacked": 0,
        "Page": 0,
        "x": -168,
        "y": 91,
        "z": 1411
    },
    # Lectern with book without name
    {
        "Book": {
        "Count": 1,
        "id": "minecraft:writable_book",
        "tag": {
            "pages": [
            "You can see strange\nthings when you're out at sea. Things that don't exist. That don't YET exist. I dreamt a spire, towering out of the seabed. I dreamt the waters falling on all sides like rain off our sails. I dreamt. I dream. I will dream."
            ]
        }
        },
        "id": "minecraft:lectern",
        "keepPacked": 0,
        "Page": 0,
        "x": 3253,
        "y": 62,
        "z": 3555
    },
    # Brewing stand with potions
    {
        "BrewTime": 0,
        "Fuel": 0,
        "id": "minecraft:brewing_stand",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:potion",
            "Slot": 1,
            "tag": {
            "display": {
                "Name": "{\"text\":\"Suspicious Potion\"}"
            },
            "Potion": "minecraft:empty",
            "RepairCost": 0
            }
        },
        {
            "Count": 1,
            "id": "minecraft:potion",
            "Slot": 2,
            "tag": {
            "Potion": "minecraft:thick"
            }
        }
        ],
        "keepPacked": 0,
        "x": -237,
        "y": 9,
        "z": 1627
    },
    # Just a tile entity, no storage (Ender Chest)
    {
        "id": "minecraft:ender_chest",
        "keepPacked": 0,
        "x": -121,
        "y": 9,
        "z": 1635
    },
    # Dispenser with tipped arrows
    {
        "id": "minecraft:dispenser",
        "Items": [
        {
            "Count": 3,
            "id": "minecraft:tipped_arrow",
            "Slot": 0,
            "tag": {
            "Potion": "minecraft:long_slowness"
            }
        },
        {
            "Count": 1,
            "id": "minecraft:tipped_arrow",
            "Slot": 1,
            "tag": {
            "Potion": "minecraft:long_slowness"
            }
        },
        {
            "Count": 4,
            "id": "minecraft:tipped_arrow",
            "Slot": 2,
            "tag": {
            "Potion": "minecraft:long_slowness"
            }
        },
        {
            "Count": 3,
            "id": "minecraft:tipped_arrow",
            "Slot": 8,
            "tag": {
            "Potion": "minecraft:long_slowness"
            }
        }
        ],
        "keepPacked": 0,
        "x": -3470,
        "y": 137,
        "z": 2212
    },
    # Sign (front with text (not all rows), back blank)
    {
        "back_text": {
        "color": "black",
        "has_glowing_text": 0,
        "messages": [
            "{\"text\":\"\"}",
            "{\"text\":\"\"}",
            "{\"text\":\"\"}",
            "{\"text\":\"\"}"
        ]
        },
        "front_text": {
        "color": "black",
        "has_glowing_text": 0,
        "messages": [
            "{\"text\":\"\"}",
            "{\"text\":\"morbillion\"}",
            "{\"text\":\"dollars\"}",
            "{\"text\":\"\"}"
        ]
        },
        "id": "minecraft:sign",
        "is_waxed": 0,
        "keepPacked": 0,
        "x": -433,
        "y": 142,
        "z": 1661
    },
    # Relic vessel in chest
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:command_block",
            "Slot": 13,
            "tag": {
            "CustomModelData": 1000014,
            "display": {
                "Lore": [
                "{\"text\":\"This simple vessel is made out of highly\"}",
                "{\"text\":\"malleable metal. Carried by the Wingmakers to\"}",
                "{\"text\":\"test the faith of Tehrmari aspirants, the\"}",
                "{\"text\":\"Aspects and Deities can easily mold it to\"}",
                "{\"text\":\"show their favor for the holder.\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Divine Transformation\",\"color\":\"green\",\"italic\":false}",
                "{\"text\":\"Can be transformed into any Relic you have\",\"color\":\"dark_gray\"}",
                "{\"text\":\"already unlocked with a being at their\",\"color\":\"dark_gray\"}",
                "{\"text\":\"devotion altar.\",\"color\":\"dark_gray\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Trinket\",\"color\":\"green\",\"italic\":false}"
                ],
                "Name": "{\"text\":\"Relic Vessel\",\"color\":\"green\",\"italic\":false,\"underlined\":true}"
            },
            "relic_vessel": 1
            }
        }
        ],
        "keepPacked": 0,
        "x": -149,
        "y": 156,
        "z": 2867
    },
    # Paper with lore and enchantment
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:paper",
            "Slot": 13,
            "tag": {
            "display": {
                "Lore": [
                "{\"text\":\"Dahr fahn Lorahn\",\"color\":\"dark_purple\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Ihb fahn Rihelma\",\"color\":\"dark_purple\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Voynath nylsh axh'malrih\",\"color\":\"dark_purple\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Ithlahr harhte\",\"color\":\"dark_purple\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Nari fahn tohsima\",\"color\":\"dark_purple\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Ertahn oulh silnar\",\"color\":\"dark_purple\"}"
                ],
                "Name": "{\"text\":\"Ancient Theocracy Rites\",\"color\":\"gold\",\"italic\":false,\"underlined\":true}"
            },
            "Enchantments": [
                {
                "id": "minecraft:protection",
                "lvl": 1
                }
            ],
            "HideFlags": 1
            }
        }
        ],
        "keepPacked": 0,
        "x": -631,
        "y": 68,
        "z": 4920
    },
    # Stone of Agony, multiple "text" in one row of Lore
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:command_block",
            "Slot": 13,
            "tag": {
            "CooldownEnch": 1,
            "CustomModelData": 1000020,
            "display": {
                "Lore": [
                "{\"text\":\"A rare stone with inordinately high potentia which\"}",
                "{\"text\":\"seems to ease your suffering. When you hold it up to\"}",
                "{\"text\":\"your ear, you can hear a single voice sobbing in a\"}",
                "{\"text\":\"storm at sea.\"}",
                "{\"text\":\" \"}",
                "[{\"text\":\"[\",\"color\":\"gray\",\"italic\":false},{\"text\":\"Ɑ\",\"color\":\"gold\",\"italic\":false},{\"text\":\"]\",\"color\":\"gray\",\"italic\":false}]"
                ],
                "Name": "{\"text\":\"Stone of Agony\",\"color\":\"gold\",\"italic\":false,\"underlined\":true}"
            },
            "Enchantments": [
                {
                }
            ],
            "MythicStone": 1
            }
        }
        ],
        "keepPacked": 0,
        "x": -614,
        "y": 83,
        "z": 4914
    },
    # Name text is in an array for some reason, like Lore text in Stone of Agony above
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:leather_boots",
            "Slot": 13,
            "tag": {
            "Damage": 0,
            "display": {
                "color": 1481884,
                "Name": "[{\"text\":\"Spirefarer's Boots\",\"italic\":false}]"
            },
            "Enchantments": [
                {
                "id": "feather_falling",
                "lvl": 4
                },
                {
                "id": "protection",
                "lvl": 2
                }
            ]
            }
        }
        ],
        "keepPacked": 0,
        "x": -1601,
        "y": 69,
        "z": 5261
    },
    # Lore text is just a string, no JSON text...
    {
        "id": "minecraft:barrel",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:barrel",
            "Slot": 13,
            "tag": {
            "BlockEntityTag": {
                "id": "minecraft:barrel",
                "Items": [
                {
                    "Count": 1,
                    "id": "minecraft:barrel",
                    "Slot": 13,
                    "tag": {
                    "display": {
                        "Name": "{\"text\":\"how\"}"
                    },
                    "RepairCost": 0
                    }
                }
                ]
            },
            "display": {
                "Lore": [
                "\"(+NBT)\""
                ],
                "Name": "{\"text\":\"how\"}"
            },
            "RepairCost": 0
            }
        }
        ],
        "keepPacked": 0,
        "x": -2918,
        "y": 71,
        "z": 5245
    },
    # Name has no text only translate
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:filled_map",
            "Slot": 24,
            "tag": {
            "Decorations": [
                {
                "id": "+",
                "rot": 180.0,
                "type": 26,
                "x": 1321.0,
                "z": -1543.0
                }
            ],
            "display": {
                "Name": "{\"translate\":\"filled_map.buried_treasure\"}"
            },
            "map": 0
            }
        },
        {
            "Count": 1,
            "id": "minecraft:filled_map",
            "Slot": 25,
            "tag": {
            "Decorations": [
                {
                "id": "+",
                "rot": 180.0,
                "type": 26,
                "x": 4921.0,
                "z": -2583.0
                }
            ],
            "display": {
                "Name": "{\"translate\":\"filled_map.buried_treasure\"}"
            },
            "map": 7
            }
        },
        {
            "Count": 1,
            "id": "minecraft:filled_map",
            "Slot": 26,
            "tag": {
            "Decorations": [
                {
                "id": "+",
                "rot": 180.0,
                "type": 26,
                "x": 2809.0,
                "z": -1063.0
                }
            ],
            "display": {
                "Name": "{\"translate\":\"filled_map.buried_treasure\"}"
            },
            "map": 6
            }
        }
        ],
        "keepPacked": 0,
        "x": 27513,
        "y": 195,
        "z": -286
    },
    # Gay Apple
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:golden_apple",
            "Slot": 13,
            "tag": {
            "CustomModelData": 1,
            "display": {
                "Name": "[\"\",{\"text\":\"Gay Apple\",\"italic\":false,\"color\":\"#FF5EFA\"}]"
            }
            }
        }
        ],
        "keepPacked": 0,
        "x": -3204,
        "y": 137,
        "z": 3038
    },
    # Totem of Dying
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:totem_of_undying",
            "Slot": 22,
            "tag": {
            "AttributeModifiers": [
                {
                "Amount": -100,
                "AttributeName": "generic.max_health",
                "Name": "generic.max_health",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    1182255616,
                    1503742804,
                    -1575955254,
                    1740454159
                ]
                }
            ],
            "CustomModelData": 1,
            "display": {
                "Lore": [
                "{\"text\":\"Nihil! Nihil! NIHIL!\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"When in offhand:\",\"color\":\"gray\"}",
                "{\"text\":\"+??? Max Health\",\"color\":\"blue\",\"italic\":false}",
                "{\"text\":\"+??? Movement Speed\",\"color\":\"blue\",\"italic\":false}",
                "{\"text\":\"+??? Attack Damage\",\"color\":\"blue\",\"italic\":false}",
                "{\"text\":\" \"}",
                "{\"text\":\"Artifact\",\"color\":\"aqua\",\"italic\":false}"
                ],
                "Name": "{\"text\":\"Totem of Dying\",\"color\":\"aqua\",\"italic\":false,\"underlined\":true}"
            },
            "HideFlags": 2
            }
        }
        ],
        "keepPacked": 0,
        "x": 2839,
        "y": 114,
        "z": -2046
    },
    # Silent Effigy (all spaces, underlined)
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:charcoal",
            "Slot": 13,
            "tag": {
            "AttributeModifiers": [
                {
                "Amount": 40.0,
                "AttributeName": "minecraft:generic.max_health",
                "Name": "minecraft:generic.max_health",
                "Operation": 0,
                "Slot": "offhand",
                "UUID": [
                    2033391086,
                    -301774224,
                    -2023809868,
                    1340251599
                ]
                },
                {
                "Amount": -0.9,
                "AttributeName": "minecraft:generic.attack_speed",
                "Name": "minecraft:generic.attack_speed",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    -72484056,
                    1144082207,
                    -2015734643,
                    -775957501
                ]
                },
                {
                "Amount": -0.9,
                "AttributeName": "minecraft:generic.attack_damage",
                "Name": "minecraft:generic.attack_damage",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    1189948442,
                    1651984696,
                    -1565862379,
                    1011205538
                ]
                },
                {
                "Amount": -0.7,
                "AttributeName": "minecraft:generic.movement_speed",
                "Name": "minecraft:generic.movement_speed",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    689791170,
                    -851884407,
                    -1488860023,
                    427597965
                ]
                }
            ],
            "CustomModelData": 1,
            "display": {
                "CustomModelData": 1,
                "Lore": [
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"...\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"gray\",\"text\":\"When in offhand:\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+40 Max Health\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"-70% Movement Speed\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"-90% Attack Damage\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"-90% Attack Speed\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"aqua\",\"text\":\"Artifact\"}],\"text\":\"\"}"
                ],
                "Name": "{\"text\":\"                 \"}"
            },
            "HideFlags": 2,
            "silentEffigy": 1
            }
        }
        ],
        "keepPacked": 0,
        "x": 4291,
        "y": 20,
        "z": 4125
    },
    # Slime Chrysalis
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:slime_ball",
            "Slot": 13,
            "tag": {
            "AttributeModifiers": [
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.max_health",
                "Name": "minecraft:generic.max_health",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    512130000,
                    848775422,
                    -2146813742,
                    2055955159
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.attack_damage",
                "Name": "minecraft:generic.attack_damage",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    -1854105528,
                    754206422,
                    -1271431418,
                    -282515582
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.movement_speed",
                "Name": "minecraft:generic.movement_speed",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    876075404,
                    -100776905,
                    -1893930755,
                    -919637245
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.attack_speed",
                "Name": "minecraft:generic.attack_speed",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    -1560919485,
                    1361857570,
                    -1392987305,
                    876324325
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.attack_knockback",
                "Name": "minecraft:generic.attack_knockback",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    1951707611,
                    1014251581,
                    -1708330939,
                    -782447642
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.knockback_resistance",
                "Name": "minecraft:generic.knockback_resistance",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    -1324298308,
                    862604695,
                    -1295976676,
                    -1149302105
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.armor",
                "Name": "minecraft:generic.armor",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    785088882,
                    -440251866,
                    -1653276020,
                    -159595343
                ]
                },
                {
                "Amount": 0.04,
                "AttributeName": "minecraft:generic.armor_toughness",
                "Name": "minecraft:generic.armor_toughness",
                "Operation": 1,
                "Slot": "offhand",
                "UUID": [
                    -363438259,
                    -1254538770,
                    -1793720045,
                    961201325
                ]
                }
            ],
            "CustomModelData": 1,
            "display": {
                "Lore": [
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"A pleasant warmth emanates off this peculiar orb.\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"While holding it, you can feel it gently writhe\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"and begin to conform to the shape of your hand.\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"It's oddly comforting.\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"gray\",\"text\":\"When in offhand:\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Max Health\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Movement Speed\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Knockback Resistance\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Attack Knockback\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Attack Damage\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Attack Speed\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Armor\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"+4% Armor Toughness\"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                "{\"extra\":[{\"italic\":false,\"color\":\"aqua\",\"text\":\"Artifact\"}],\"text\":\"\"}"
                ],
                "Name": "{\"text\":\"Slime Chrysalis\",\"color\":\"aqua\",\"italic\":false,\"underlined\":true}"
            },
            "HideFlags": 2
            }
        }
        ],
        "keepPacked": 0,
        "x": -1225,
        "y": 93,
        "z": 3752
    },
    # Divine Bauble + Enchanted book
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:iron_nugget",
            "Slot": 1,
            "tag": {
            "CustomModelData": 3,
            "display": {
                "Lore": [
                "{\"text\":\"This long multicolored tassel seems to\"}",
                "{\"text\":\"enjoy wrapping itself around sturdy\"}",
                "{\"text\":\"objects and swaying, leaving small runes\"}",
                "{\"text\":\"in the air behind it as it moves.\"}",
                "{\"text\":\" \"}",
                "{\"text\":\"Can be exchanged for valuables with\"}",
                "{\"text\":\"Precocious Kinah in Ytaj.\"}"
                ],
                "Name": "{\"text\":\"Divine Bauble\",\"color\":\"green\",\"italic\":false,\"underlined\":true}"
            }
            }
        },
        {
            "Count": 1,
            "id": "minecraft:enchanted_book",
            "Slot": 12,
            "tag": {
            "StoredEnchantments": [
                {
                "id": "minecraft:sweeping",
                "lvl": 1
                }
            ]
            }
        }
        ],
        "keepPacked": 0,
        "x": -147,
        "y": 103,
        "z": -1690
    },
    # Dahr pearl
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:command_block",
            "Slot": 13,
            "tag": {
            "CustomModelData": 1007005,
            "dahr_pearl": 1,
            "display": {
                "Lore": [
                "{\"text\":\"This alabaster gem is slightly flattened on one end.\"}"
                ],
                "Name": "{\"text\":\"White Pearl\",\"color\":\"white\",\"italic\":false}"
            },
            "Enchantments": [
                {
                }
            ],
            "pearlID": 6
            }
        }
        ],
        "keepPacked": 0,
        "x": -205,
        "y": 5,
        "z": 1143
    },
    # Player Head (with Name)
    {
        "id": "minecraft:chest",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:repeating_command_block",
            "Slot": 11
        },
        {
            "Count": 1,
            "id": "minecraft:paper",
            "Slot": 13,
            "tag": {
            "display": {
                "Name": "{\"text\":\"Who would win?\",\"italic\":\"false\"}"
            }
            }
        },
        {
            "Count": 1,
            "id": "minecraft:player_head",
            "Slot": 15,
            "tag": {
            "display": {
                "Name": "{\"text\":\"Some Cringe Nerd\",\"italic\":\"false\"}"
            },
            "SkullOwner": {
                "Id": [
                -129729193,
                -1125498227,
                -2010196209,
                -950668269
                ],
                "Name": "Zerdguyyy",
                "Properties": {
                "textures": [
                    {
                    "Signature": "mlSxddzUiK1P874a07T9ckuR/ltxkCKjMo0bPOSCU/0gXqX0C+YaDsYYQtPhQ3JElgGyHcRe41pfvPZBRcWscW3nb5AswJ87v8cfhegeOSb/xrlFzar023i0X0QG0kexOTKPytincsYmdpKpjF3hLjqsc0HcOujOzZQqYlDFAjL3K9r7XxQnqFW2kAcUxF8yZ755xjnbSBu37nKHUpdHlfpAPshTXuqmHGKCjvs/H/8YxIs+eo1HDDSF0KR0aaj26EgyYbnK2g4IDb+y1HnJ+xuaLkcM3DobetTGfWFJJ7mkJfcHVhjrI0dufenexI226RPfJb41AYqGb43W2xFK+6OfMJIvTs3ombzc6AFg7XVa8/mLxxlNkr93B2SqNfdIUiYTJJIYVHqIgtjaYp9yUMZ/oEf06+jly69ME7+RQ6nd75hsho0hiAkRC3pEc7/C2+wMq5uzImFhezThzsoe3ArEZ5qgrRDOKGegAXXogCZNoYqMj0p0xeGUmOwQRbTj9fBDmEA54uZDEi5p5TLMsGqWfsB17eA+z5/lqfPZ1ytYdUs+7jZKifamYgBW82QpF1FtGowdQ3NMcgLdvbn7d+62HQ0XdkCUsLYs44Eu5DYETZdQ/v6yx2cZfktC+VrH/iHBGkyNOzfZn9kHjh5JWoOqEOMbKlJX2SKFIEyv8Ks=",
                    "Value": "ewogICJ0aW1lc3RhbXAiIDogMTY4NDM4OTIyOTAwMCwKICAicHJvZmlsZUlkIiA6ICJmODQ0N2Q1N2JjZWE0MjhkODgyZWQ3MGZjNzU1ZjQxMyIsCiAgInByb2ZpbGVOYW1lIiA6ICJaZXJkZ3V5eXkiLAogICJzaWduYXR1cmVSZXF1aXJlZCIgOiB0cnVlLAogICJ0ZXh0dXJlcyIgOiB7CiAgICAiU0tJTiIgOiB7CiAgICAgICJ1cmwiIDogImh0dHA6Ly90ZXh0dXJlcy5taW5lY3JhZnQubmV0L3RleHR1cmUvNTYyZWIwODU5OTFhNTJhMmZlNmVlMmQyYWUwZGM1N2YwYjZhYWUwZGFiN2ZkOWVmOWI3ZmYwOTVmZTM1MjIxOSIsCiAgICAgICJtZXRhZGF0YSIgOiB7CiAgICAgICAgIm1vZGVsIiA6ICJzbGltIgogICAgICB9CiAgICB9CiAgfQp9"
                    }
                ]
                }
            }
            }
        }
        ],
        "keepPacked": 0,
        "x": 27550,
        "y": 108,
        "z": -649
    },
    # Splash Potion of Water?
    {
        "BrewTime": 0,
        "Fuel": 0,
        "id": "minecraft:brewing_stand",
        "Items": [
        {
            "Count": 1,
            "id": "minecraft:splash_potion",
            "Slot": 2,
            "tag": {
            "Potion": "minecraft:water"
            }
        }
        ],
        "keepPacked": 0,
        "x": 3865,
        "y": 98,
        "z": 3480
    },
    #
]
print(len(test_json))

23


In [8]:
@dataclass
class NameType:
    """Class for keeping track of a name and type for a Minecraft object."""
    name: str
    mc_type: str

    def __iter__(self):
        return iter(astuple(self))

def mc_id_to_name(id: str) -> str:
    """Remove 'minecraft:' from id."""
    return id[10:]

COMPASS_TERMINAL = {
    "Network Terminal Locator 0xAVS",
    "Network Terminal Locator 0xSMV",
    "Network Terminal Locator 0xEXC",
    "Network Terminal Locator 0xADM",
    "Core Security Checkpoint Locator",
    "Primal Energy Radar"
    }

DREHMAL_TO_NAME_TYPE = {
    "RunicCatalyst": NameType("Runic Catalyst", "runic_catalyst"),
    "khive_scroll": NameType("Khivian Scroll of Sanctuary", "khive_scroll"),
    "relic_vessel": NameType("Relic Vessel", "relic_vessel"),
    "Parenchyma": NameType("Parenchyma", "parenchyma"),
    "Shade": NameType("Penumbra", "penumbra"),
    "CrystalClaw": NameType("Crystal Digging Claws", "claw"),
    "Osteo": NameType("Osteogenesis", "osteo"),
    "whispersong": NameType("Whispersong", "whispersong"),
    "CooldownEnch": NameType("Stone of Agony", "stone_of_agony"),
    "VitalityEnch": NameType("Stone of Luxury", "stone_of_luxury"),
    "SpeedEnch": NameType("Stone of Worry", "stone_of_worry"),
    "AvPod": NameType("AvPod", "avpod"),
    "virtuo_aegis": NameType("Eyebiter", "eyebiter"),
    "ExodusTank": NameType("Tank Keyfob", "keyfob"),
    "avHorseArmor": NameType("WarpHorse Armor MkIII", "warp_horse_armor"),
    "AvHorseRemote": NameType("WarpHorse Receiver MkII", "warphorse_receiver"),
    "Voidtear": NameType("Voidtear Dagger", "voidtear"),
    "Flammer": NameType("Flammer", "flammer"),
    "Anyrs_Sceptre": NameType("Emperor Anyr's Scepter", "anyr_scepter"),
    "UltvaBowblade": NameType("Ultva's Bowblade", "bowblade"),
    "Heartaxe": NameType("The Heartaxe", "heartaxe"),
    "PureCorruption": NameType("Pure Corruption", "pure_corruption"),
    "Scars": NameType("One Thousand Scars", "scars"),
    "Orchid": NameType("Orchidaceae", "orchidaceae"),
    "Platemail": NameType("Rhentite Plate Mail", "rehntite_chestplate"),
    "Frostfang": NameType("The Frostfang", "frostfang"),
    "Glider" : NameType("Avsohm'Kohl", "elytra"),
    "Hovadhammer": NameType("Hovadchear's Greathammer", "hovad"),
    "Masayoshi": NameType("Masayoshi", "masayoshi"),
    "tul_v": NameType("Tul'Vohaln", "tulvohaln"),
    "PeaceTreaty": NameType("Peace Treaty", "peace_treaty"),
    "FesteringStrides": NameType("Festering Strides", "festering_strides"),
    "Aeongale": NameType("Aeongale", "aeongale"),
    "proxigea": NameType("Proxigea", "proxigea"),
    "tcrux": NameType("Thundercrux", "thundercrux"),
    "Magestep": NameType("Magestep", "magestep"),
    "WardStaff": NameType("Aurastaff of Permafrost", "aurastaff_item"),
    "pris_mace": NameType("Call of the Council", "prismatic_mace"),
    "runic_amplifier": NameType("Runic Amplifier", "mcguffin"),
    # this is error items, there are two purple_dye trades that have custom models.
    # The trades are inaccessible and the items look bugged.
    "Furn": NameType("FurnItem", "invalid_item"),
}

type GroupMC = Literal["storage", "lectern", "item_frame", "sign", "trader", "armor_stand", "entity", "block_entity"]

class MCItem(TypedDict, total=False):
    name: str
    displayName: str
    count: int
    lore: str | None
    enchanted: bool
    pages: list[str] | None

class MCTrade(TypedDict):
    buy: MCItem
    buyB: MCItem | None
    sell: MCItem

class MCSign(TypedDict):
    text_front: list[str] | None
    text_back: list[str] | None
    color_front: str
    color_back: str

class MCEntity(TypedDict, total=False):
    x: int
    y: int
    z: int
    dim: str
    name: str
    displayName: str
    group: GroupMC
    items: list[MCItem] | None
    trades: list[MCTrade] | None
    sign: MCSign | None


DREHMAL_ITEMS = {"RunicCatalyst", "khive_scroll", "relic_vessel", "Parenchyma", "Shade",
                 "CrystalClaw", "Osteo", "whispersong", "CooldownEnch", "VitalityEnch", "SpeedEnch",
                 "AvPod", "virtuo_aegis", "ExodusTank", "avHorseArmor", "AvHorseRemote", "Voidtear",
                 "Flammer", "Anyrs_Sceptre", "UltvaBowblade", "Heartaxe", "PureCorruption", "Scars",
                 "Orchid", "Platemail", "Frostfang", "Glider", "Hovadhammer", "Masayoshi",
                 "tul_v", "PeaceTreaty", "FesteringStrides",
                 "Aeongale", "proxigea", "tcrux", "Magestep", "WardStaff", "pris_mace",
                 "runic_amplifier", "Furn"}
# Necroblade, AK47, silentEffigy, NihilistNotes

TODO
- ✅Totem of Dying = minecraft:totem_of_undying
- ✅Divine Bauble (lodahr)
- ✅Runic Amplifier (mcguffin)
- Stasis Bolt (lodahr)
- Elder Mead (lodahr)
- ✅Call of the Council (maybe only a trade)
- ✅Aurastaff of Permafrost
- ✅Nail (A trade)
- ✅Dahr Perls (6)
---
- Festering Strides (Drop from Warden Entity?)


### Item parsing and other
- Text
- Compass
- Potions
- Drehmal items

In [9]:
type TextFormattedDictLine = dict[str, bool|str]  # Line with formatting
type TextDict = dict[str, str|TextFormattedDictLine|list[TextFormattedDictLine]]
type TextArray = list[str|TextFormattedDictLine]

def text_from_dict(text_dict: TextDict) -> str|None:
    """
    Get text from a dictionary, no formatting is returned.

    There can be some confusing text dictionaries
    Example: {"extra":[{"italic":True,
                        "color":"dark_purple",
                        "text":"..."}],
              "text":""
             }
    OR: {"text":"Some text here","other_formatting":str|bool}

    :param text_dict: Minecraft TextDict
    :return: String
    """
    if "extra" in text_dict:
        # This must be caused by some conversion from 1.17 to 1.20.1 because the 'text' in `text_dict`
        # is always "" and the real text is in the 'extra' array
        return text_dict["extra"][0]["text"]
    elif "text" in text_dict:
        # The 1.17 and normal way of getting text
        return text_dict["text"]
    else:
        return None


def translate_text_override(text_line_unparsed: str) -> str:
    if "block.minecraft.ominous_banner" in text_line_unparsed:
        return "Ominous Banner"
    elif "filled_map.buried_treasure" in text_line_unparsed:
        return "Buried Treasure Map"
    elif "entity.minecraft.killer_bunny" in text_line_unparsed:
        return "The Killer Bunny"
    else:
        print("'text' did not exist in line data and does not have correction in code\n"
              "This is likely due to only having 'translate' key."
              f"text_line= {text_line_unparsed}")
        raise KeyError("Provide override to `translate_text_override`")


def text_from_array(text_array: TextArray) -> str:
    """
    There can be some confusing text arrays
    Example: ["",
              {"text":"Gay Apple",
               "italic":False,
               "color":"#FF5EFA"},
              {"extra": [...], "text": "..."},
             ]
             OR
             [
                ['some text here'],
                ['text', {}],...
             ]

    :param text_array: Minecraft text array of strings and/or dicts
    :return: String
    """
    match text_array:
        case [str(), dict()]:
            # Starts with an empty string
            text_idx_1 = text_from_dict(text_array[1])
            if text_idx_1 is None:
                raise KeyError("text is missing and part of a list[str, dict], IDK?")

            if text_array[0] == "":
                return text_idx_1
            else:
                # Starts with something else, I am unsure if this happens
                return f"{text_array[0]}\n{text_idx_1}"
        case [str()]:
            # Single line of text in the array
            return text_array[0]
        case [str(), dict(), str()]:
            # There is a single case of this in v2.2.2 MC1.20.1
            return f"{text_array[0]}{text_from_dict(text_array[1])}{text_array[2]}"
        case [*dicts] if all(isinstance(d, dict) for d in dicts) and dicts:
            # All text dicts, treat each one as a separate line
            segments = []
            for d_segment in text_array:
                text = text_from_dict(d_segment)
                if text is None:
                    raise KeyError("text is missing and part of a list[dicts], IDK?")
                segments.append(text)
            return "".join(segments)

        case _:
            print(f"Error, unknown text array: {text_array}")
            return "Unknown text array"

def parse_single_line_text(text_line: str) -> str:
    # Parse the json text line into a dictionary with python types
    line_data = json.loads(text_line)

    # if isinstance(line_data, dict):
    #     # Its only 1 dict, get text directly
    #     try:
    #         return line_data["text"]
    #     except KeyError as e:
    #         if "block.minecraft.ominous_banner" in text_line:
    #             return "Ominous Banner"
    #         elif "filled_map.buried_treasure" in text_line:
    #             return "Buried Treasure Map"
    #         elif "entity.minecraft.killer_bunny" in text_line:
    #             return "The Killer Bunny"
    #         else:
    #             e.add_note("'text' did not exist in line data and does not have correction in code\n"
    #                        "This is likely due to only having 'translate' key."
    #                        f"text_line= {text_line}")
    #             raise
    # elif isinstance(line_data, list):
    #     # There can be a list of dicts because different parts of the line have different formatting
    #     # get each segment separately then join before adding to full_text
    #     segments = []
    #     for text_segment in line_data:
    #         # in 2.2.2 (1.20.1) it is possible to get a list of the form [\"\", {"text": "..."}]
    #         if isinstance(text_segment, dict):
    #             segments.append(text_segment["text"])
    #         elif isinstance(text_segment, str):
    #             segments.append(text_segment)
    #         else:
    #             # Just in case something weird happens
    #             raise TypeError(f"Unexpected type {type(text_segment)} in {line_data=}")
    #     return "".join(segments)
    if isinstance(line_data, dict):
        text = text_from_dict(line_data)
        if text is None:
            # Special cases
            return translate_text_override(text_line)
        return text
    elif isinstance(line_data, list):
        return text_from_array(line_data)
    elif isinstance(line_data, str):
        return line_data
    else:
        # Unknown
        raise NotImplementedError(f"This type is not implemented in parse_mc_text: {type(line_data)}\n{line_data}")

def parse_mc_text(text: str | list[str] | None) -> str | None:
    """
    Parse Minecraft Text.

    :param text:
    :return: A single string without formatting or json items or None
    """
    if text is None:
        return None

    if isinstance(text, str):
        return parse_single_line_text(text)
    elif isinstance(text, list):
        full_text = []
        for line in text:
            full_text.append(parse_single_line_text(line))
        return "\n".join(full_text)
    else:
        raise ValueError(f"Expect string, list of strings or None, got {type(text)}")

### Custom Icons
There are many custom items with custom textures and/or models. Legendary, Mythical, and some others have special data for this but some just have custom model data and a name. The below function is used to deal with those cases so they are assigned the custom texture later by setting the name to be the same as the texture, not the original minecraft item name.
Items include:
- coiled_nail, nail, pale_ore
- burnt_apple
- dirty_paper
- divine_bauble
- elden_ring
- gay_apple
- slime_chrysalis
- rusty_sword
- fearsrtiker
- talisman_growth
- vihktor_spoils
- totemofdying

In [10]:
def special_item_overrides(id_: str,
                           display_name: str, count: int,
                           enchanted: bool, lore: str) -> MCItem | None:
    if display_name == "Gay Apple":
        # Special Case for Gay Apple
        return MCItem(name="gay_apple", displayName="Gay Apple", count=count,
                      lore=None, enchanted=False)
    if display_name == "Totem of Dying":
        return MCItem(name="totemofdying", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "                 ":
        # silentEffigy (has all spaces in name)
        return MCItem(name="charcoal", displayName="_________________",
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Ther Erldern Rirng":
        return MCItem(name="elden_ring", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Dirty Paper" and id_ == "paper":
        return MCItem(name="dirty_paper", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Talisman of Endless Growth" and id_ == "green_dye":
        return MCItem(name="talisman_growth", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Axe of Growth":
        return MCItem(name="growth_axe", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Rusty Sword" and id_ == "iron_sword":
        return MCItem(name="rusty_sword", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "fearsrtiker" and id_ == "crossbow":
        # Misspelling is intentional
        return MCItem(name="fearsrtiker", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Vikhtor's Spoils":
        return MCItem(name="vihktor_spoils", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Nail" and id_ == "iron_sword":
        # Sold by a villager
        return MCItem(name="nail", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Coiled Nail" and id_ == "iron_sword":
        return MCItem(name="coiled_nail", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Pale Ore":
        return MCItem(name="pale_ore", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Slime Chrysalis":
        return MCItem(name="slime_chrysalis", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    if display_name == "Divine Bauble" and id_ == "iron_nugget":
        return MCItem(name="divine_bauble", displayName=display_name,
                      count=count, lore=lore, enchanted=enchanted)
    return None

In [11]:
def find_drehmal_item(item: dict) -> MCItem | None:
    item = [i for i in DREHMAL_ITEMS if i in item]
    if len(item) == 0:
        return None
    return item[0]

def potion_object_name(potion_name: str, id_: str) -> str:
    if id_ == "potion" and potion_name in NO_EFFECT_POTIONS:
        return MC_POTION_NAME_TO_DISPLAY[potion_name]

    try:
        prefix = POTION_PREFIX[id_]
    except KeyError:
        print(f"No potion prefix match in json file for {id_}")
        prefix = "Object of "
    
    try:
        display_name = prefix + MC_POTION_NAME_TO_DISPLAY[potion_name]
    except KeyError:
        print(f"No potion match in json file for {potion_name}")
        display_name = prefix + "-ERROR- Potion"

    return display_name

def special_compass_name(compass_display_name: str) -> str:
    """Get the name of the unique compass texture depending on display name.
    If it's not found, returns the default: compass.

    :param compass_display_name: compass display name
    :return: compass texture name
    """
    if compass_display_name == "Compass of Nihility":
        return "compass_nihility"
    elif compass_display_name == "Snake Ornamented Compass":
        return "compass_snake"
    elif compass_display_name == "Lotus Shaped Compass":
        return "compass_lotus"
    elif compass_display_name in COMPASS_TERMINAL:
        return "compass_term"
    elif compass_display_name == "Resonant Compass":  # found in Dusps graveyard
        return "compass_yav"
    elif compass_display_name == "Ytaj Compass":  # unused?
        return "compass_ytaj"
    else:  # default
        return "compass"

def set_dahr_pearls(pearl_id: int) -> str:
    """Get the ``name`` for the colored pearls from Dahr's challenge.

    :param pearl_id: id number
    :return: string that matches texture name
    """
    match pearl_id:
        case 1:
            return "pearl_black"
        case 2:
            return "pearl_blue"
        case 3:
            return "pearl_yellow"
        case 4:
            return "pearl_red"
        case 5:
            return "pearl_grey"
        case 6:
            return "pearl_white"
        case _:
            print(f"Invalid pearl_id: {pearl_id}, this should not exist?")
            return "xxxx"

def parse_item(item: dict, has_slots: bool = True) -> MCItem:
    name = None
    display_name = None
    lore = None
    enchanted = False
    pages_text = None
    count = item["Count"]
    # Use to check if something has its own model/texture in this function
    custom_model_pass_check = True

    id_ = mc_id_to_name(item["id"])

    if len(item) == 3 and has_slots:
        # Standard item, nothing special
        name = id_
        display_name = MC_NAME_TO_DISPLAY[name]
        return MCItem(name=name, displayName=display_name, count=count,
                      lore=lore, enchanted=enchanted)
    
    if id_ == "bundle":
        # Don't need lore for bundle
        return MCItem(name="bundle", displayName="Bundle", count=count,
                      lore=None, enchanted=False)

    # Check for extra data found in 'tag'
    if tag := item.get("tag"):
        if "Enchantments" in tag:
            # Enchanted Item
            enchanted = True
        if "StoredEnchantments" in tag:
            # Enchanted Book
            enchanted = True
        if "CustomModelData" in tag:
            custom_model_pass_check = False
        
        if display := tag.get("display"):
            display_name = parse_mc_text(display.get("Name"))
            lore = parse_mc_text(display.get("Lore"))
            if id_ == "compass":
                # Special Cases for Compass variants
                return MCItem(name=special_compass_name(display_name), displayName=display_name,
                              count=count, lore=lore, enchanted=enchanted)
            if pearl_id := tag.get("pearlID"):
                # Dahr challenge pearls
                pearl_name = set_dahr_pearls(pearl_id)
                return MCItem(name=pearl_name, displayName=display_name,
                              count=count, lore=lore, enchanted=enchanted)

            # Special Overrides
            # Use when an item has model data or something else special to make sure it has the correct texture.
            # This is not for items that have an attribute/tag in their data with their type
            #  (EX: legendary/mythicals, Runic Catalyst, Khive Scroll, Runic Amplifier, ect)
            #  These should be added to `DREHMAL_TO_NAME_TYPE` and `DREHMAL_ITEMS`
            item_special_override = special_item_overrides(id_, display_name, count, enchanted, lore)
            if item_special_override is not None:
                return item_special_override

        if pages := tag.get("pages"):
            # Just count the pages, too much effort to convert all this text.
            lore = f"{len(pages)} Pages"
            pages_text = pages
        if ditem := find_drehmal_item(tag):
            display_name, name = DREHMAL_TO_NAME_TYPE[ditem]
            custom_model_pass_check = True
        if "Potion" in tag:
            # Get potion name instead of just "Potion"
            if display_name is None:
                display_name = potion_object_name(tag["Potion"], id_)
    
    if not name:
        # If `name` has not been found yet check for special Drehmal data not in 'tag' first
        if ditem := find_drehmal_item(item):
            display_name, name = DREHMAL_TO_NAME_TYPE[ditem]
            custom_model_pass_check = True
        else:
            # just a regular MC item
            name = id_
            if not display_name:
                display_name = MC_NAME_TO_DISPLAY[name]

    if not custom_model_pass_check:
        if id_ == "command_block" and display_name == "Command Block" and tag is not None:
            if tag.get("CustomModelData") == 33333333:
                # 27 128x128 distorted Luigi image command blocks in Mt. Yavhlix door animation room.
                return MCItem(name=name, displayName="green.png Command Block", count=count, lore=lore, enchanted=enchanted, pages=pages_text)
        print(f"⚠️: failed to set unique `name` for something with custom texture: {id_=}, N={name}, DN={display_name},\n"
              f"   Item={item}")

    return MCItem(name=name, displayName=display_name, count=count,
                  lore=lore, enchanted=enchanted, pages=pages_text)


### Block/Tile Entities

In [12]:
def combine_dict_items(items: list[dict]) -> list[dict]:
    combined = {}
    
    # This will merge things that have the same name but enchanted/non-enchanted and lore/no-lore
    for item in items:
        display_name = item['displayName']
        if display_name in combined:
            combined[display_name]['count'] += item['count']
        else:
            combined[display_name] = item
    
    return list(combined.values())

def count_and_list_items(items_list: list[dict]) -> list[MCItem]:
    reduced_item_list = []
    for item in items_list:
        # Get list of items
        reduced_item_list.append(parse_item(item))
    
    # Sum up same item counts
    if len(items_list) <= 1:
        return reduced_item_list
    else:
        return combine_dict_items(reduced_item_list)

def extract_sign_data(obj_sign: dict) -> MCSign|None:
    """Create the sign data for the front and back of the sign."""
    text_front = [parse_single_line_text(text) for text in obj_sign["front_text"]["messages"]]
    if text_front == BLANK_SIGN_FRONT_DATA:
        text_front = None
    text_back = [parse_single_line_text(text) for text in obj_sign["back_text"]["messages"]]
    if text_back == BLANK_SIGN_DATA:
        text_back = None
    if text_front is None and text_back is None:
        # We don't care about empty signs
        return None
    return MCSign(text_front=text_front, text_back=text_back,
                  color_front=obj_sign["front_text"]["color"],
                  color_back=obj_sign["back_text"]["color"])

def block_entity_with_items(obj: dict, dimension: str) -> MCEntity:
    name = mc_id_to_name(obj["id"])
    display_name = MC_NAME_TO_DISPLAY[name]
    items = count_and_list_items(obj["Items"])
    return {
        "x": obj["x"],
        "y": obj["y"],
        "z": obj["z"],
        "dim": dimension,
        "name": name,
        "displayName": display_name,
        "group": "storage",
        "items": items,
    }

def block_entity_lectern(obj: dict, dimension: str) -> MCEntity:
    name = mc_id_to_name(obj["id"])
    display_name = MC_NAME_TO_DISPLAY[name]
    items = [parse_item(obj["Book"], has_slots=False)]
    return {
        "x": obj["x"],
        "y": obj["y"],
        "z": obj["z"],
        "dim": dimension,
        "name": "Lectern_with_Book",  # Allows for special icon instead of empty lectern
        "displayName": display_name,
        "group": "lectern",
        "items": items
    }

def block_entity_sign(obj: dict, dimension: str) -> MCEntity|None:
    name = mc_id_to_name(obj["id"])
    # display_name = MC_NAME_TO_DISPLAY[name]
    # We can't know what wood type the sign is because it is not part of the block entity.
    sign_data = extract_sign_data(obj)
    if sign_data is not None:
        # Only if there is text data for at least one side of the sign
        return MCEntity(
            x=obj["x"],
            y=obj["y"],
            z=obj["z"],
            dim=dimension,
            name=name,
            displayName="Sign",
            group="sign",
            sign=sign_data
        )
    return None

def block_entity_basic(obj: dict, dimension: str) -> MCEntity:
    name = mc_id_to_name(obj["id"])
    display_name = MC_NAME_TO_DISPLAY[name]
    return {
        "x": obj["x"],
        "y": obj["y"],
        "z": obj["z"],
        "dim": dimension,
        "name": name,
        "displayName": display_name,
        "group": "block_entity"
    }

def extract_data_block_entities(block_entities: list[dict], dimension: str) -> list[MCEntity]:
    simplified_block_entity_list = []
    for be in block_entities:
        if be.get("Items"):
            simplified_block_entity_list.append(block_entity_with_items(be, dimension))
        elif be.get("Book"):  # a lectern with a book
            simplified_block_entity_list.append(block_entity_lectern(be, dimension))
        elif be.get("front_text"):  # a sign with text
            sign_data = block_entity_sign(be, dimension)
            if sign_data is not None:
                # We don't want signs that are blank on both sides.
                simplified_block_entity_list.append(sign_data)
        else:
            simplified_block_entity_list.append(block_entity_basic(be, dimension))
    
    return simplified_block_entity_list

In [25]:
test_be_list = extract_data_block_entities(test_json, "overworld")
print("Item Count:", len(test_be_list))
test_be_list

Item Count: 23


[{'x': -192,
  'y': 73,
  'z': -347,
  'dim': 'overworld',
  'name': 'chest',
  'displayName': 'Chest',
  'group': 'storage',
  'items': [{'name': 'runic_catalyst',
    'displayName': 'Runic Catalyst',
    'count': 1,
    'lore': 'A small, magical orb valued by\ntraders and arcanists. They have\nseveral applications in both\nmagical creations and technology.',
    'enchanted': False,
    'pages': None}]},
 {'x': -361,
  'y': 108,
  'z': 769,
  'dim': 'overworld',
  'name': 'chest',
  'displayName': 'Chest',
  'group': 'storage',
  'items': [{'name': 'bread',
    'displayName': 'Bread',
    'count': 2,
    'lore': None,
    'enchanted': False},
   {'name': 'paper',
    'displayName': 'Paper',
    'count': 4,
    'lore': None,
    'enchanted': False},
   {'name': 'compass',
    'displayName': 'Compass',
    'count': 1,
    'lore': None,
    'enchanted': False}]},
 {'x': -504,
  'y': 71,
  'z': 946,
  'dim': 'overworld',
  'name': 'chest',
  'displayName': 'Chest',
  'group': 'storage',
 

## Entities
There is a ton of data per entity.

#### Important Values
- "id"
- "Pos": The location, should probably clip to an int value. An array of 3 numbers
- "CustomName"
- "Offers" -> for villager, wandering_trader
    - "Recipes" ->
        - "buy" & "buyB" & "sell" (should skip minecraft:air / Count=0)
            - This is an item with minimum 2 properties
                - "Count"
                - "id"
            - Consider skipping lore for some items (bundle, runic catalysts)
            - They can also have "tag" with extra properties (like the tile entity items but without "Slot")
- "ArmorItems" but only the ones that have Lore/Name (like Warden, Smau)
    - Exception: `minecraft:armor_stand` show all "ArmorItems"
- `minecraft:chest_minecart` if it has "Items" (treat like chest then) they might not have "Items" at all and only have "LootTable" (skip these or do something else)
    - Almost all only have "LootTable", **Skip these**
- All other properties can be skipped

In [14]:
TEST_ENTITIES_JSON = [
    # Adventuring Merchant (wandering_trader)
    {
        "DeathTime": 0,
        "DespawnDelay": 0,
        "LeftHanded": 0,
        "OnGround": 1,
        "AbsorptionAmount": 0.0,
        "Attributes": [{"Name": "minecraft:generic.movement_speed","Base": 0.699999988079071}],
        "Invulnerable": 1,
        "Brain": {"memories": {}},
        "Age": 0,
        "HandDropChances": [0.08500000089406967, 0.08500000089406967],
        "ArmorDropChances": [0.08500000089406967, 0.08500000089406967, 0.08500000089406967, 0.08500000089406967],
        "Rotation": [-90.0, 0.0],
        "HurtByTimestamp": 0,
        "ForcedAge": 0,
        "CustomName": "{\"text\":\"Adventuring Merchant\"}",
        "ArmorItems": [{},{},{},{}],
        "Air": 300,
        "HandItems": [{"id": "minecraft:filled_map","Count": 1},{}],
        "NoAI": 1,
        "Offers": {
            "Recipes": [
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 10
                    },
                    "sell": {
                        "id": "minecraft:filled_map",
                        "tag": {
                            "display": {
                                "Name": "{\"text\":\"Lorahn'Kahl Map\",\"italic\":false}",
                                "MapColor": 5764351,
                                "Lore": [
                                    "{\"text\":\"A map of the region surrounding\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"Mohta, with markers \",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"signifying important locations.\",\"color\":\"dark_purple\"}"
                                ]
                            },
                            "map": 1002
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:air",
                        "Count": 0
                    }
                },
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 45
                    },
                    "sell": {
                        "id": "minecraft:filled_map",
                        "tag": {
                            "display": {
                                "Name": "{\"text\":\"Map of Drehmal\",\"italic\":false}",
                                "MapColor": 3290191,
                                "Lore": [
                                    "{\"text\":\"A map of the entire continent of\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"Drehmal, showing the locations\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"of its towns, major rivers,\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"coastlines, and more.\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\" \"}",
                                    "{\"text\":\"Towns & Cities:\",\"color\":\"gray\",\"italic\":false}",
                                    "{\"text\":\"New Drabyel\",\"color\":\"green\"}",
                                    "{\"text\":\"Okeke\",\"color\":\"yellow\"}",
                                    "{\"text\":\"Ebonrun\",\"color\":\"red\"}",
                                    "{\"text\":\"Athrah\",\"color\":\"gold\"}",
                                    "{\"text\":\"Fort Nimahj\",\"color\":\"dark_blue\"}",
                                    "{\"text\":\"Tharxax\",\"color\":\"dark_red\"}",
                                    "{\"text\":\"Mohta\",\"color\":\"aqua\"}",
                                    "{\"text\":\"Gozak\",\"color\":\"dark_green\"}",
                                    "{\"text\":\"Firteid\",\"color\":\"dark_aqua\"}",
                                    "{\"text\":\"Mossfield\",\"color\":\"blue\"}",
                                    "{\"text\":\"Highfall\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"Dusps\",\"color\":\"light_purple\"}"
                                ]
                            },
                            "map": 103
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:diamond",
                        "Count": 1
                    }
                },
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 32
                    },
                    "sell": {
                        "id": "minecraft:shulker_box",
                        "tag": {
                            "display": {
                                "Name": "{\"text\":\"Runic Vessel\",\"italic\":false}",
                                "Lore": [
                                    "{\"text\":\"An arcane crate dotted with\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"magical runes. It can be used as\",\"color\":\"dark_purple\"}",
                                    "{\"text\":\"a portable storage device.\",\"color\":\"dark_purple\"}"
                                ]
                            },
                            "BlockEntityTag": {
                                "id": "minecraft:shulker_box",
                                "CustomName": "{\"text\":\"Runic Vessel\"}"
                            }
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:command_block",
                        "tag": {
                            "display": {
                                "Name": "{\"text\":\"Runic Catalyst\",\"color\":\"aqua\",\"italic\":false}",
                                "Lore": [
                                    "{\"text\":\"A small, magical orb valued by\"}",
                                    "{\"text\":\"traders and arcanists. They have\"}",
                                    "{\"text\":\"several applications in both\"}",
                                    "{\"text\":\"magical creations and technology.\"}"
                                ]
                            },
                            "CustomModelData": 1000000,
                            "RunicCatalyst": 1
                        },
                        "Count": 12
                    }
                },
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 20
                    },
                    "sell": {
                        "id": "minecraft:bundle",
                        "tag": {
                            "display": {
                                "Lore": [
                                    "{\"text\":\"Can store up to 64 different stackable items.\"}",
                                    "{\"text\":\"While in inventory, drag and right click items\"}",
                                    "{\"text\":\"onto bundle to store inside. Right click to take\"}",
                                    "{\"text\":\"most recently stored item out of bundle.\"}",
                                    "{\"text\":\"Crouch and right click while in hand to throw\"}",
                                    "{\"text\":\"out all stored items.\"}"
                                ]
                            }
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:air",
                        "Count": 0
                    }
                },
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 3
                    },
                    "sell": {
                        "id": "minecraft:lead",
                        "Count": 2
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:air",
                        "Count": 0
                    }
                }
            ]
        },
        "UUID": [
            -21036722,
            1846496517,
            -1888776485,
            222472065
        ],
        "Inventory": [],
        "FallDistance": 0.0,
        "NoGravity": 1,
        "id": "minecraft:wandering_trader",
        "Motion": [0.0,0.0,0.0],
        "Fire": 0,
        "Pos": [-64.5,65.0,5295.5],
        "CanPickUpLoot": 0,
        "Health": 20.0,
        "HurtTime": 0,
        "FallFlying": 0,
        "PersistenceRequired": 1,
        "PortalCooldown": 0
    },
    # Warden (has legendary item equipped, drops probably with custom DeathLootTable)
    {
        "DeathTime": 0,
        "DeathLootTable": "entities:mob/warden",
        "LeftHanded": 0,
        "OnGround": 1,
        "AbsorptionAmount": 0.0,
        "IsBaby": 0,
        "Attributes": [
            {"Name": "minecraft:generic.max_health","Base": 80.0},
            {"Name": "minecraft:generic.knockback_resistance","Base": 0.8},
            {"Name": "minecraft:generic.movement_speed","Base": 0.2},
            {"Name": "minecraft:generic.armor_toughness","Base": 0.0},
            {"Name": "minecraft:generic.attack_damage","Base": 7.0},
            {"Name": "minecraft:generic.armor","Base": 2.0}
        ],
        "Invulnerable": 0,
        "Brain": {"memories": {}},
        "ActiveEffects": [
            {
                "ShowIcon": 1,
                "ShowParticles": 1,
                "Id": 10,
                "Duration": 199998938,
                "Ambient": 0,
                "Amplifier": 1
            }
        ],
        "HandDropChances": [-999.0,0.08500000089406967],
        "ArmorDropChances": [999.0,-999.0,-999.0,-999.0],
        "Rotation": [270.20025634765625,2.454364538192749],
        "HurtByTimestamp": 0,
        "CanBreakDoors": 0,
        "CustomName": "{\"text\":\"Shatterhorn Warden\"}",
        "InWaterTime": -1,
        "ArmorItems": [
            {
                "id": "minecraft:iron_boots",
                "tag": {
                    "festeringStrides": 1,
                    "Enchantments": [
                        {"id": "minecraft:unbreaking","lvl": 5},
                        {"id": "minecraft:mending","lvl": 1}
                    ],
                    "Damage": 0,
                    "HideFlags": 2,
                    "FesteringStrides": 1,
                    "display": {
                        "Name": "{\"text\":\"Festering Strides\",\"color\":\"light_purple\",\"italic\":false,\"underlined\":true}",
                        "Lore": [
                            "{\"text\":\"The warden of Shatterhorn Gulch was\"}",
                            "{\"text\":\"a terrible, sadistic man of gluttony.\"}",
                            "{\"text\":\"Some time ago a powerful arcanist\"}",
                            "{\"text\":\"was imprisoned within the tallest\"}",
                            "{\"text\":\"belfry, who, in their grand escape,\"}",
                            "{\"text\":\"cursed the warden with immortality\"}",
                            "{\"text\":\"and endless festering wounds.\"}",
                            "{\"text\":\" \"}",
                            "{\"text\":\"Necrotic Vigor\",\"color\":\"light_purple\",\"italic\":false}",
                            "{\"text\":\"Wearing these boots grants the user\",\"color\":\"dark_gray\"}",
                            "{\"text\":\"a regenerating absorption shield.\",\"color\":\"dark_gray\"}",
                            "{\"text\":\" \"}",
                            "{\"text\":\"When on feet:\",\"color\":\"gray\",\"italic\":false}",
                            "{\"text\":\"+10 Max Health\",\"color\":\"blue\",\"italic\":false}",
                            "{\"text\":\"-40% Armor\",\"color\":\"blue\",\"italic\":false}",
                            "{\"text\":\"-40% Armor Toughness\",\"color\":\"blue\",\"italic\":false}",
                            "{\"text\":\" \"}",
                            "{\"text\":\"Legendary\",\"color\":\"light_purple\",\"italic\":false}"
                        ]
                    },
                    "Legendary": 1,
                    "CustomModelData": 1000420,
                    "AttributeModifiers": [
                        {
                            "Amount": 10,
                            "Operation": 0,
                            "Slot": "feet",
                            "UUID": [
                                -554014167,
                                -1522121882,
                                -2106057169,
                                1473545412
                            ],
                            "AttributeName": "generic.max_health",
                            "Name": "generic.max_health"
                        },
                        {
                            "Amount": -0.4,
                            "Operation": 1,
                            "Slot": "feet",
                            "UUID": [
                                -1221841741,
                                21317117,
                                -1323745472,
                                -1171662616
                            ],
                            "AttributeName": "generic.armor",
                            "Name": "generic.armor"
                        },
                        {
                            "Amount": -0.4,
                            "Operation": 1,
                            "Slot": "feet",
                            "UUID": [
                                -1143731544,
                                -1270004280,
                                -1345022317,
                                1140235342
                            ],
                            "AttributeName": "generic.armor_toughness",
                            "Name": "generic.armor_toughness"
                        }
                    ]
                },
                "Count": 1
            },
            {
                "id": "minecraft:chainmail_leggings",
                "tag": {
                    "Damage": 0,
                    "Enchantments": [
                        {"id": "minecraft:thorns","lvl": 1}
                    ]
                },
                "Count": 1
            },
            {
                "id": "minecraft:iron_chestplate",
                "tag": {
                    "Damage": 0,
                    "Enchantments": [
                        {"id": "minecraft:thorns","lvl": 1}
                    ]
                },
                "Count": 1
            },
            {
                "id": "minecraft:player_head",
                "tag": {
                    "SkullOwner": {
                        "Id": [
                            -1068105783,
                            1192971976,
                            -1627142992,
                            -190737827
                        ],
                        "Properties": {
                            "textures": [
                                {
                                    "Value": "eyJ0ZXh0dXJlcyI6eyJTS0lOIjp7InVybCI6Imh0dHA6Ly90ZXh0dXJlcy5taW5lY3JhZnQubmV0L3RleHR1cmUvODk2OWZmMTdkOWZjMzQ5ZDRhMzA1YjA2NTYzNjAyOGRiOTAwZTAzMjIzMjdkMmM0N2Q2ZjYyNjJmMmIwZDBmMSJ9fX0="
                                }
                            ]
                        }
                    }
                },
                "Count": 1
            }
        ],
        "Tags": ["warden"],
        "HandItems": [
            {
                "id": "minecraft:iron_axe",
                "tag": {"Damage": 0},
                "Count": 1
            },
            {}
        ],
        "Air": 300,
        "UUID": [-1049845167,1680883878,-1654397158,-1819891181],
        "DrownedConversionTime": -1,
        "FallDistance": 0.0,
        "id": "minecraft:zombie",
        "Motion": [0.0,-0.0784000015258789,0.0],
        "Fire": -1,
        "Pos": [4651.197437354053,104.0625,-737.5015689800342],
        "CanPickUpLoot": 0,
        "Health": 88.0,
        "HurtTime": 0,
        "FallFlying": 0,
        "PersistenceRequired": 1,
        "PortalCooldown": 0
    },
    # Armor stand with 1 item
    {
        "DeathTime": 0,
        "OnGround": 1,
        "AbsorptionAmount": 0.0,
        "Attributes": [
            {
                "Name": "minecraft:generic.armor",
                "Base": 0.0
            },
            {
                "Name": "minecraft:generic.armor_toughness",
                "Base": 0.0
            },
            {
                "Name": "minecraft:generic.movement_speed",
                "Base": 0.699999988079071
            }
        ],
        "Invulnerable": 0,
        "Brain": {"memories": {}},
        "Pose": {
            "Head": [1.989734172821045,-7.224034309387207,0.0],
            "Body": [0.0,.473031997680664,0.0]
        },
        "Rotation": [-90.0,0.0],
        "HurtByTimestamp": 0,
        "ArmorItems": [{},{},
            {
                "id": "minecraft:chainmail_chestplate",
                "tag": {
                    "Damage": 0
                },
                "Count": 1
            },
            {}
        ],
        "Invisible": 0,
        "Air": 300,
        "HandItems": [{},{}],
        "UUID": [-1583043160,-1250801326,-1759837558,-1649671957],
        "NoBasePlate": 0,
        "FallDistance": 0.0,
        "id": "minecraft:armor_stand",
        "Motion": [0.0,-0.0784000015258789,0.0],
        "Pos": [-1292.5,54.0,447.5],
        "Health": 20.0,
        "DisabledSlots": 0,
        "HurtTime": 0,
        "ShowArms": 0,
        "FallFlying": 0,
        "Fire": -1,
        "PortalCooldown": 0,
        "Small": 0
    },
    # Empty Armor stand
    {
        "DeathTime": 0,
        "OnGround": 1,
        "AbsorptionAmount": 0.0,
        "Attributes": [
            {
                "Name": "minecraft:generic.movement_speed",
                "Base": 0.699999988079071
            }
        ],
        "Invulnerable": 0,
        "Brain": {"memories": {}},
        "Pose": {
            "Head": [2.215264081954956,-0.7019319534301758,0.0],
            "Body": [0.0,-1.8589849472045898,0.0]
        },
        "Rotation": [-90.0,0.0],
        "HurtByTimestamp": 0,
        "ArmorItems": [{},{},{},{}],
        "Invisible": 0,
        "Air": 300,
        "HandItems": [{},{}],
        "UUID": [-62895262,102779508,-1088515882,-578258162],
        "NoBasePlate": 0,
        "FallDistance": 0.0,
        "id": "minecraft:armor_stand",
        "Motion": [0.0,-0.0784000015258789,0.0],
        "Pos": [-484.5,74.0,-1949.5],
        "Health": 20.0,
        "DisabledSlots": 0,
        "HurtTime": 0,
        "ShowArms": 0,
        "FallFlying": 0,
        "Fire": -1,
        "PortalCooldown": 0,
        "Small": 0
    },
    # Chest Minecart with LootTable
    {
        "LootTable": "minecraft:chests/abandoned_mineshaft",
        "Motion": [0.0,0.0,0.0],
        "Invulnerable": 0,
        "Air": 300,
        "OnGround": 0,
        "PortalCooldown": 0,
        "Rotation": [0.0, 0.0],
        "FallDistance": 0.0,
        "Pos": [25730.5,25.0625,-320.5],
        "Fire": -1,
        "id": "minecraft:chest_minecart",
        "UUID": [-1989773531,-960950950,1016694289,-1438388068],
        "LootTableSeed": -8382892626075341017
    },
    # Chest Minecart with Items (1 here)
    {
        "OnGround": 0,
        "Air": 300,
        "UUID": [-259333447,-1502919397,-1957390072,-1206262593],
        "Invulnerable": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:chest_minecart",
        "Rotation": [0.0,0.0],
        "Items": [
            {
                "Slot": 13,
                "id": "minecraft:command_block",
                "tag": {
                    "display": {
                        "Name": "{\"text\":\"Runic Catalyst\",\"color\":\"aqua\",\"italic\":false}",
                        "Lore": [
                            "{\"text\":\"A small, magical orb valued by\"}",
                            "{\"text\":\"traders and arcanists. They have\"}",
                            "{\"text\":\"several applications in both\"}",
                            "{\"text\":\"magical creations and technology.\"}"
                        ]
                    },
                    "CustomModelData": 1000000,
                    "RunicCatalyst": 1
                },
                "Count": 1
            }
        ],
        "Pos": [-1806.5099999904633,119.0625,-878.5],
        "Fire": -1,
        "PortalCooldown": 0
    },
    # Runic Nailsmith (sells Nail)
    {
        "DeathTime": 0,
        "RestocksToday": 0,
        "Xp": 2,
        "LeftHanded": 0,
        "OnGround": 0,
        "AbsorptionAmount": 0.0,
        "FoodLevel": 0,
        "LastRestock": 0,
        "Attributes": [ {"Name": "minecraft:generic.movement_speed","Base": 0.5}],
        "Invulnerable": 1,
        "Brain": {"memories": {}},
        "Age": 0,
        "HandDropChances": [0.08500000089406967,0.08500000089406967],
        "ArmorDropChances": [0.08500000089406967,0.08500000089406967,0.08500000089406967,0.08500000089406967],
        "Rotation": [-90.0,0.0],
        "HurtByTimestamp": 0,
        "ForcedAge": 0,
        "CustomName": "{\"text\":\"Runic Nailsmith\"}",
        "ArmorItems": [{},{},{},{}],
        "Air": 300,
        "HandItems": [{},{}],
        "NoAI": 1,
        "Offers": {
            "Recipes": [
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:emerald",
                        "Count": 28
                    },
                    "sell": {
                        "id": "minecraft:iron_sword",
                        "tag": {
                            "RepairCost": 10000000,
                            "Enchantments": [
                                {
                                    "id": "minecraft:bane_of_arthropods",
                                    "lvl": 6
                                },
                                {
                                    "id": "minecraft:sweeping",
                                    "lvl": 3
                                },
                                {
                                    "id": "minecraft:unbreaking",
                                    "lvl": 3
                                }
                            ],
                            "Damage": 0,
                            "HideFlags": 2,
                            "Nail": 1,
                            "display": {
                                "Name": "{\"text\":\"Nail\",\"color\":\"yellow\",\"italic\":false,\"underlined\":true}",
                                "Lore": [
                                    "{\"text\":\"A traditional weapon from a land much smaller\"}",
                                    "{\"text\":\"than our own. The hilt is inscribed with\"}",
                                    "{\"text\":\"unfamiliar, insect-shaped carvings.\"}",
                                    "{\"text\":\" \"}",
                                    "{\"text\":\"When in main hand:\",\"color\":\"gray\",\"italic\":false}",
                                    "{\"text\":\"9 Attack Damage\",\"color\":\"blue\",\"italic\":false}",
                                    "{\"text\":\"1.8 Attack Speed\",\"color\":\"blue\",\"italic\":false}",
                                    "{\"text\":\" \"}",
                                    "{\"text\":\"Artisan\",\"color\":\"yellow\",\"italic\":false}"
                                ]
                            },
                            "CustomModelData": 2,
                            "AttributeModifiers": [
                                {
                                    "Amount": 8,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -413408944,
                                        -991476836,
                                        -2061813768,
                                        -1813037337
                                    ],
                                    "AttributeName": "generic.attack_damage",
                                    "Name": "generic.attack_damage"
                                },
                                {
                                    "Amount": -2.2,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -1459487241,
                                        -164082472,
                                        -1937655763,
                                        169122015
                                    ],
                                    "AttributeName": "generic.attack_speed",
                                    "Name": "generic.attack_speed"
                                }
                            ]
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 9999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:quartz_block",
                        "Count": 1
                    }
                },
                {
                    "xp": 1,
                    "buy": {
                        "id": "minecraft:iron_sword",
                        "tag": {
                            "RepairCost": 10000000,
                            "Enchantments": [
                                {
                                    "id": "minecraft:bane_of_arthropods",
                                    "lvl": 6
                                },
                                {
                                    "id": "minecraft:sweeping",
                                    "lvl": 3
                                },
                                {
                                    "id": "minecraft:unbreaking",
                                    "lvl": 3
                                }
                            ],
                            "Damage": 0,
                            "HideFlags": 2,
                            "Nail": 1,
                            "display": {
                                "Name": "{\"text\":\"Nail\",\"color\":\"yellow\",\"italic\":false,\"underlined\":true}",
                                "Lore": [
                                    "{\"text\":\"A traditional weapon from a land much smaller\"}",
                                    "{\"text\":\"than our own. The hilt is inscribed with\"}",
                                    "{\"text\":\"unfamiliar, insect-shaped carvings.\"}",
                                    "{\"text\":\" \"}",
                                    "{\"text\":\"When in main hand:\",\"color\":\"gray\",\"italic\":false}",
                                    "{\"text\":\"9 Attack Damage\",\"color\":\"blue\",\"italic\":false}",
                                    "{\"text\":\"1.8 Attack Speed\",\"color\":\"blue\",\"italic\":false}",
                                    "{\"text\":\" \"}",
                                    "{\"text\":\"Artisan\",\"color\":\"yellow\",\"italic\":false}"
                                ]
                            },
                            "CustomModelData": 2,
                            "AttributeModifiers": [
                                {
                                    "Amount": 8,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -413408944,
                                        -991476836,
                                        -2061813768,
                                        -1813037337
                                    ],
                                    "AttributeName": "generic.attack_damage",
                                    "Name": "generic.attack_damage"
                                },
                                {
                                    "Amount": -2.2,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -1459487241,
                                        -164082472,
                                        -1937655763,
                                        169122015
                                    ],
                                    "AttributeName": "generic.attack_speed",
                                    "Name": "generic.attack_speed"
                                }
                            ]
                        },
                        "Count": 1
                    },
                    "sell": {
                        "id": "minecraft:iron_sword",
                        "tag": {
                            "RepairCost": 10000000,
                            "Enchantments": [
                                {
                                    "id": "minecraft:bane_of_arthropods",
                                    "lvl": 6
                                },
                                {
                                    "id": "minecraft:sweeping",
                                    "lvl": 5
                                },
                                {
                                    "id": "minecraft:mending",
                                    "lvl": 1
                                },
                                {
                                    "id": "minecraft:unbreaking",
                                    "lvl": 3
                                }
                            ],
                            "Damage": 0,
                            "HideFlags": 2,
                            "Nail": 1,
                            "display": {
                                "Name": "{\"extra\":[{\"italic\":false,\"underlined\":true,\"color\":\"yellow\",\"text\":\"Coiled Nail\"}],\"text\":\"\"}",
                                "Lore": [
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"A traditional weapon from a land much smaller\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"than our own. The hilt is inscribed with\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"unfamiliar, insect-shaped carvings.\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":false,\"color\":\"gray\",\"text\":\"When in main hand:\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"13 Attack Damage\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":false,\"color\":\"blue\",\"text\":\"1.8 Attack Speed\"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\" \"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":false,\"color\":\"yellow\",\"text\":\"Artisan\"}],\"text\":\"\"}"
                                ]
                            },
                            "CustomModelData": 11,
                            "AttributeModifiers": [
                                {
                                    "Amount": 12.0,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -413408944,
                                        -991476836,
                                        -2061813768,
                                        -1813037337
                                    ],
                                    "AttributeName": "minecraft:generic.attack_damage",
                                    "Name": "minecraft:generic.attack_damage"
                                },
                                {
                                    "Amount": -2.2,
                                    "Operation": 0,
                                    "Slot": "mainhand",
                                    "UUID": [
                                        -1459487241,
                                        -164082472,
                                        -1937655763,
                                        169122015
                                    ],
                                    "AttributeName": "minecraft:generic.attack_speed",
                                    "Name": "minecraft:generic.attack_speed"
                                }
                            ]
                        },
                        "Count": 1
                    },
                    "uses": 0,
                    "priceMultiplier": 0.0,
                    "maxUses": 9999999,
                    "rewardExp": 0,
                    "demand": 0,
                    "specialPrice": 0,
                    "buyB": {
                        "id": "minecraft:command_block",
                        "tag": {
                            "display": {
                                "Name": "{\"extra\":[{\"italic\":false,\"color\":\"aqua\",\"text\":\"Pale Ore\"}],\"text\":\"\"}",
                                "Lore": [
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"A small and odd-looking metal \"}],\"text\":\"\"}",
                                    "{\"extra\":[{\"italic\":true,\"color\":\"dark_purple\",\"text\":\"that is unusually heavy.\"}],\"text\":\"\"}"
                                ]
                            },
                            "CustomModelData": 1000023
                        },
                        "Count": 1
                    }
                }
            ]
        },
        "UUID": [1687241164,894716694,-1167634613,184963458],
        "Gossips": [],
        "Inventory": [],
        "VillagerData": {
            "type": "minecraft:plains",
            "profession": "minecraft:weaponsmith",
            "level": 99
        },
        "FallDistance": 0.0,
        "NoGravity": 1,
        "id": "minecraft:villager",
        "Motion": [0.0,0.0,0.0],
        "Fire": 0,
        "Pos": [4459.5,119.0,-3273.5],
        "CanPickUpLoot": 1,
        "Health": 20.0,
        "HurtTime": 0,
        "FallFlying": 0,
        "PersistenceRequired": 1,
        "LastGossipDecay": 1138817272,
        "PortalCooldown": 0
    },
    # Item frame simple item
    {
        "Invisible": 1,
        "ItemDropChance": 1.0,
        "Item": {"id": "minecraft:book","Count": 1},
        "ItemRotation": 3,
        "OnGround": 0,
        "Air": 300,
        "UUID": [-36253519,-1599126628,-2053664686,888693436],
        "Invulnerable": 0,
        "Fixed": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:item_frame",
        "Rotation": [0.0,-90.0],
        "Facing": 1,
        "TileZ": -564,
        "Pos": [-171.5,164.03125,-563.5],
        "Fire": -1,
        "PortalCooldown": 0,
        "TileY": 164,
        "TileX": -172
    },
    # Item frame complex item
    {
        "Invisible": 1,
        "ItemDropChance": 1.0,
        "Item": {
            "id": "minecraft:writable_book",
            "tag": {
                "pages": [
                    "The warmth of Lai is  simply pitiful, worthy of nothing but scorn. The foolish worm blathers about basic idiocies like \"kindness\" and \"compassion\"; does he not know the perils we face? Our hated enemy, that of the cold, encroaches every day. No. Only we blessed few see the true light, the light of ",
                    "the Torahn! Obviously! Lai cannot compete with the cruel heat of our sacred sphere. We have seen it, communed with it, and tested out mettle; we shall be victorious! The naive idiots back in Merijool who still trust in Lai shall rue the day they banished the Astorahnni!"
                ],
                "display": {"Name": "{\"text\":\"Creed of the Order of Astorahn\"}"},
                "RepairCost": 0
            },
            "Count": 1
        },
        "ItemRotation": 3,
        "OnGround": 0,
        "Air": 300,
        "UUID": [1622023087,916406924,-1301667556,-62164966],
        "Invulnerable": 0,
        "Fixed": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:item_frame",
        "Rotation": [0.0,-90.0],
        "Facing": 1,
        "TileZ": -565,
        "Pos": [-172.5,180.03125,-564.5],
        "Fire": -1,
        "PortalCooldown": 0,
        "TileY": 180,
        "TileX": -173
    },
    # Item frame with Runic Catalyst
    {
        "Invisible": 1,
        "Tags": ["nbt","nbt_check"],
        "ItemDropChance": 1.0,
        "Item": {
            "id": "minecraft:command_block",
            "tag": {
                "CustomModelData": 1000000,
                "display": {
                    "Name": "{\"text\":\"Runic Catalyst\",\"color\":\"aqua\",\"italic\":false}",
                    "Lore": [
                        "{\"text\":\"A small, magical orb valued by\"}",
                        "{\"text\":\"traders and arcanists. They have\"}",
                        "{\"text\":\"several applications in both\"}",
                        "{\"text\":\"magical creations and technology.\"}"
                    ]
                },
                "RunicCatalyst": 1
            },
            "Count": 1
        },
        "ItemRotation": 0,
        "OnGround": 0,
        "Air": 300,
        "UUID": [391651469,-1614985793,-1432434747,1587118331],
        "Invulnerable": 0,
        "Fixed": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:item_frame",
        "Rotation": [180.0,0.0],
        "Facing": 2,
        "TileZ": -74,
        "Pos": [2871.5,225.5,-73.03125],
        "Fire": -1,
        "PortalCooldown": 0,
        "TileY": 225,
        "TileX": 2871
    },
    # Item frame with no item
    {
        "Tags": ["nbt_check"],
        "Invisible": 0,
        "OnGround": 0,
        "Air": 300,
        "UUID": [-1454377179,671893437,-1129481854,-852562580],
        "Invulnerable": 0,
        "Fixed": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:item_frame",
        "Rotation": [0.0,-90.0],
        "Facing": 1,
        "TileZ": -1950,
        "Pos": [-471.5,74.03125,-1949.5],
        "Fire": -1,
        "PortalCooldown": 0,
        "TileY": 74,
        "TileX": -472
    },
    # Glow Item Frame with book with array of arrays text
    {
        "Invisible": 0,
        "ItemDropChance": 1.0,
        "Item": {
            "id": "minecraft:book",
            "tag": {
                "display": {
                    "Name": "[\"\",{\"text\":\"Mercenary's Resignation\",\"italic\":false,\"color\":\"dark_red\",\"underlined\":true}]",
                    "Lore": [
                        "[\"That's it. I've reached my limit with poking around these\"]",
                        "[\"miserable old ruins! There's no sign of \",{\"text\":\"brewing stands\",\"color\":\"gold\"}]",
                        "[\"that were used by \\\"Khive's Chosen\\\". I don't see so much\"]",
                        "[\"as a \",{\"text\":\"single rune above ground\",\"color\":\"gold\"},\" in this sorry mess! This\"]",
                        "[\"is the last straw. I'm out of here, and I'm never taking\"]",
                        "[\"another job at Ancehl Castle from that slimy coot!\"]"
                    ]
                }
            },
            "Count": 1
        },
        "ItemRotation": 1,
        "OnGround": 0,
        "Air": 300,
        "UUID": [-537878923,-1175370272,-2121583626,793942492],
        "Invulnerable": 0,
        "Fixed": 0,
        "FallDistance": 0.0,
        "Motion": [0.0,0.0,0.0],
        "id": "minecraft:glow_item_frame",
        "Rotation": [0.0,-90.0],
        "Facing": 1,
        "TileZ": 1091,
        "Pos": [1118.5,85.03125,1091.5],
        "Fire": -1,
        "PortalCooldown": 0,
        "TileY": 85,
        "TileX": 1118
    },
    # Pillager with Ominous Banner that uses translate instead of text, this is skipped though because it has no Name
    {
        "Patrolling": 0,
        "DeathTime": 0,
        "CanJoinRaid": 1,
        "LeftHanded": 0,
        "OnGround": 1,
        "AbsorptionAmount": 0.0,
        "Attributes": [
            {
                "Name": "minecraft:generic.follow_range",
                "Modifiers": [
                    {
                        "Amount": 0.06782531344316105,
                        "UUID": [1797813420,-421248028,-1879002596,-1353262977],
                        "Name": "Random spawn bonus",
                        "Operation": 1
                    }
                ],
                "Base": 12.0
            },
            {"Name": "minecraft:generic.attack_damage","Base": 5.0},
            {"Name": "minecraft:generic.movement_speed","Base": 0.3499999940395355}
        ],
        "PatrolLeader": 1,
        "Invulnerable": 0,
        "Brain": {"memories": {}},
        "Rotation": [309.4490966796875,0.0],
        "HurtByTimestamp": 0,
        "ArmorDropChances": [0.08500000089406967,0.08500000089406967,0.08500000089406967,2.0],
        "HandDropChances": [0.08500000089406967,0.08500000089406967],
        "ArmorItems": [{},{},{},
            {
                "id": "minecraft:white_banner",
                "tag": {
                    "display": {
                        "Name": "{\"color\":\"gold\",\"translate\":\"block.minecraft.ominous_banner\"}"
                    },
                    "HideFlags": 32,
                    "BlockEntityTag": {
                        "id": "minecraft:banner",
                        "Patterns": [
                            {"Pattern": "mr","Color": 9},
                            {"Pattern": "bs","Color": 8},
                            {"Pattern": "cs","Color": 7},
                            {"Pattern": "bo","Color": 8},
                            {"Pattern": "ms","Color": 15},
                            {"Pattern": "hh","Color": 8},
                            {"Pattern": "mc","Color": 8},
                            {"Pattern": "bo","Color": 15}
                        ]
                    }
                },
                "Count": 1
            }
        ],
        "Air": 300,
        "HandItems": [{"id": "minecraft:iron_axe","tag": {"Damage": 0},"Count": 1},{}],
        "UUID": [-848108170,1311787952,-2053345644,-1213115982],
        "Wave": 0,
        "FallDistance": 0.0,
        "id": "minecraft:vindicator",
        "Motion": [0.0,-0.0784000015258789,0.0],
        "Fire": -1,
        "Pos": [-2125.794847525333,118.0,-139.7457347282574],
        "CanPickUpLoot": 0,
        "Health": 24.0,
        "HurtTime": 0,
        "FallFlying": 0,
        "PersistenceRequired": 1,
        "PortalCooldown": 0
    },
    # avSYS Exchange Unit villager in space
    {
        "AbsorptionAmount": 0.0,
        "ActiveEffects": [{"Ambient": 0,"Amplifier": 1,"Duration": 198865058,"Id": 14,"ShowIcon": 0,"ShowParticles": 0}],
        "Age": 0,
        "Air": 300,
        "ArmorDropChances": [0.0850000008940697,0.0850000008940697,0.0850000008940697,0.0850000008940697],
        "ArmorItems": [{},{},{},
            {
                "Count": 1,
                "id": "minecraft:player_head",
                "tag": {
                    "SkullOwner": {
                        "Id": [-959530201,-1494858482,-1712108741,-1580350872],
                        "Properties": {
                            "textures": [{"Value": "eyJ0ZXh0dXJlcyI6eyJTS0lOIjp7InVybCI6Imh0dHA6Ly90ZXh0dXJlcy5taW5lY3JhZnQubmV0L3RleHR1cmUvNDY1ODdkZDU5M2IxNTM2ZjNkNWFkZmU4YThjOTBiY2Q4NDJiNDIyN2RhOGYxMjQ3ZjE0YjgzODBkODE1NzZlNyJ9fX0="}]}
                    }
                }
            }
        ],
        "Attributes": [{"Base": 0.5,"Name": "minecraft:generic.movement_speed"}],
        "Brain": {"memories": {}},
        "CanPickUpLoot": 1,
        "CustomName": "{\"text\":\"avSYS Exchange Unit\"}",
        "DeathTime": 0,
        "FallDistance": 0.0,
        "FallFlying": 0,
        "Fire": 0,
        "FoodLevel": 0,
        "ForcedAge": 0,
        "Gossips": [],
        "HandDropChances": [0.0850000008940697,0.0850000008940697],
        "HandItems": [{},{}],
        "Health": 20.0,
        "HurtByTimestamp": 0,
        "HurtTime": 0,
        "id": "minecraft:villager",
        "Inventory": [],
        "Invulnerable": 1,
        "LastGossipDecay": 1139429047,
        "LastRestock": 0,
        "LeftHanded": 0,
        "Motion": [0.0,0.0,0.0],
        "NoAI": 1,
        "NoGravity": 1,
        "Offers": {
            "Recipes": [
                {
                    "buy": {
                        "Count": 3,
                        "id": "minecraft:iron_block"
                    },
                    "buyB": {
                        "Count": 1,
                        "id": "minecraft:diamond"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 1,
                        "id": "minecraft:command_block",
                        "tag": {
                        "CustomModelData": 1000000,
                        "display": {
                            "Lore": [
                            "{\"text\":\"A small, magical orb valued by\"}",
                            "{\"text\":\"traders and arcanists. They have\"}",
                            "{\"text\":\"several applications in both\"}",
                            "{\"text\":\"magical creations and technology.\"}"
                            ],
                            "Name": "{\"text\":\"Runic Catalyst\",\"color\":\"aqua\",\"italic\":false}"
                        },
                        "RunicCatalyst": 1
                        }
                    },
                    "specialPrice": 0,
                    "uses": 8,
                    "xp": 1
                },
                {
                    "buy": {
                        "Count": 3,
                        "id": "minecraft:diamond"
                    },
                    "buyB": {
                        "Count": 32,
                        "id": "minecraft:prismarine"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 1,
                        "id": "minecraft:heart_of_the_sea"
                    },
                    "specialPrice": 0,
                    "uses": 0,
                    "xp": 1
                },
                {
                    "buy": {
                        "Count": 3,
                        "id": "minecraft:rotten_flesh"
                    },
                    "buyB": {
                        "Count": 1,
                        "id": "minecraft:ender_pearl"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 3,
                        "id": "minecraft:phantom_membrane"
                    },
                    "specialPrice": 0,
                    "uses": 0,
                    "xp": 1
                },
                {
                "buy": {
                        "Count": 48,
                        "id": "minecraft:fire_charge"
                    },
                    "buyB": {
                        "Count": 1,
                        "id": "minecraft:poppy"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 1,
                        "id": "minecraft:wither_rose"
                    },
                    "specialPrice": 0,
                    "uses": 0,
                    "xp": 1
                },
                {
                    "buy": {
                        "Count": 1,
                        "id": "minecraft:wither_rose"
                    },
                    "buyB": {
                        "Count": 32,
                        "id": "minecraft:emerald"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 1,
                        "id": "minecraft:wither_skeleton_skull"
                    },
                    "specialPrice": 0,
                    "uses": 0,
                    "xp": 1
                },
                {
                    "buy": {
                        "Count": 14,
                        "id": "minecraft:emerald"
                    },
                    "buyB": {
                        "Count": 1,
                        "id": "minecraft:glass_bottle"
                    },
                    "demand": 0,
                    "maxUses": 999999,
                    "priceMultiplier": 0.0,
                    "rewardExp": 1,
                    "sell": {
                        "Count": 1,
                        "id": "minecraft:experience_bottle"
                    },
                    "specialPrice": 0,
                    "uses": 0,
                    "xp": 1
                }
            ]
        },
        "OnGround": 0,
        "PersistenceRequired": 1,
        "PortalCooldown": 0,
        "Pos": [-32.5,141.75,97.5],
        "RestocksToday": 0,
        "Rotation": [0.0,0.0],
        "Silent": 1,
        "UUID": [842329251,-704035492,-1723827236,-983522280],
        "VillagerData": {"level": 99,"profession": "minecraft:none","type": "minecraft:plains"},
        "Xp": 8
    },
]
print(len(TEST_ENTITIES_JSON))

14


In [13]:
def _entity_display_name(entity: dict, name: str) -> str:
    if custom_name := entity.get("CustomName"):
        return parse_mc_text(custom_name)
    else:
        return MC_NAME_TO_DISPLAY[name]

def _create_entity_trade(recipe: dict[str, Any]) -> MCTrade:
    """Create a trade dictionary given a recipe

    :param recipe: Recipe dictionary with 'buy', 'buyB', and 'sell'
    :return: MCTrade dictionary
    """
    buy = parse_item(recipe["buy"], has_slots=False)
    if data_buyB := recipe.get("buyB"):
        if data_buyB["Count"] == 0 or data_buyB["id"] == "minecraft:air":
            buyB = None
        else:
            buyB = parse_item(recipe["buyB"], has_slots=False)
    else:
        buyB = None
    sell = parse_item(recipe["sell"], has_slots=False)
    return MCTrade(buy=buy, buyB=buyB, sell=sell)

def entity_armor_stand(entity: dict, dimension: str) -> MCEntity:
    # can they hold items in hands? I am assuming no.
    name = mc_id_to_name(entity["id"])
    display_name = _entity_display_name(entity, name)

    # Armor Stand won't have duplicate items and a check before this made sure there was at least 1 item
    item_list = []
    for item in entity["ArmorItems"]:
        if item:
            # Get list of items
            item_list.append(parse_item(item))
    
    return {
        "x": int(entity["Pos"][0]),
        "y": int(entity["Pos"][1]),
        "z": int(entity["Pos"][2]),
        "dim": dimension,
        "name": name,
        "displayName": display_name,
        "group": "armor_stand",
        "items": item_list
    }

def entity_tradeable(entity: dict, dimension: str) -> MCEntity:
    name = mc_id_to_name(entity["id"])
    display_name = _entity_display_name(entity, name)

    trades = []
    trade: dict[str, Any]
    for trade in entity["Offers"]["Recipes"]:
        trades.append(_create_entity_trade(trade))

    return {
        "x": int(entity["Pos"][0]),
        "y": int(entity["Pos"][1]),
        "z": int(entity["Pos"][2]),
        "dim": dimension,
        "name": name,
        "displayName": display_name,
        "group": "trader",
        "trades": trades,
    }

def entity_other(entity: dict, dimension: str, item_type: Literal["Items", "Item", "none"]) -> MCEntity:
    name = mc_id_to_name(entity["id"])
    display_name = _entity_display_name(entity, name)

    if item_type == "Items":
        # already checked for 1 or more Items it now behaves like a block_entity with slots
        item_list = count_and_list_items(entity["Items"])
        group = "storage"
    elif item_type == "Item":
        # already checked Item existing
        item_list = [parse_item(entity["Item"], has_slots=False)]
        group = "item_frame"
    else:
        item_list = None
        group = "entity"

    return {
        "x": int(entity["Pos"][0]),
        "y": int(entity["Pos"][1]),
        "z": int(entity["Pos"][2]),
        "dim": dimension,
        "name": name,
        "displayName": display_name,
        "group": group,
        "items": item_list
    }

def extract_data_entities(entities: list[dict], dimension: str) -> list[MCEntity]:
    simplified_entity_list = []
    for ent in entities:
        ent_id = ent.get("id")
        if ent_id in SKIP_ENTITIES:
            continue
        elif ent_id == "minecraft:armor_stand" and any(ent["ArmorItems"]):
            simplified_entity_list.append(entity_armor_stand(ent, dimension))
        elif ent_id in ENTITY_WITH_ITEMS and ent.get("Items") and any(ent["Items"]):
            simplified_entity_list.append(entity_other(ent, dimension, "Items"))
        elif ent_id in ENTITY_WITH_SINGLE_ITEM and ent.get("Item"):
            simplified_entity_list.append(entity_other(ent, dimension, "Item"))
        elif ent.get("CustomName"):  # Named entity
            if ent.get("Offers") and len(ent["Offers"].get("Recipes")) >= 1:
                simplified_entity_list.append(entity_tradeable(ent, dimension))
            else:
                simplified_entity_list.append(entity_other(ent, dimension, "none"))
        else:
            # Skip normal mobs, including villagers
            continue
    
    return simplified_entity_list

In [16]:
test_entity_list = extract_data_entities(TEST_ENTITIES_JSON, "overworld")
print(len(test_entity_list))
test_entity_list

10


[{'x': -64,
  'y': 65,
  'z': 5295,
  'dim': 'overworld',
  'name': 'wandering_trader',
  'displayName': 'Adventuring Merchant',
  'group': 'trader',
  'trades': [{'buy': {'name': 'emerald',
     'displayName': 'Scale',
     'count': 10,
     'lore': None,
     'enchanted': False,
     'pages': None},
    'buyB': None,
    'sell': {'name': 'filled_map',
     'displayName': "Lorahn'Kahl Map",
     'count': 1,
     'lore': 'A map of the region surrounding\nMohta, with markers \nsignifying important locations.',
     'enchanted': False,
     'pages': None}},
   {'buy': {'name': 'emerald',
     'displayName': 'Scale',
     'count': 45,
     'lore': None,
     'enchanted': False,
     'pages': None},
    'buyB': {'name': 'diamond',
     'displayName': 'Diamond',
     'count': 1,
     'lore': None,
     'enchanted': False,
     'pages': None},
    'sell': {'name': 'filled_map',
     'displayName': 'Map of Drehmal',
     'count': 1,
     'lore': 'A map of the entire continent of\nDrehmal, sho

---
# Combine Tile and Entity into 1 object
1. Loop over all dimensions, getting block/tile_entity and entity lists
2. combine all lists into 1
3. save to single JSON

In [14]:
def get_all_data() -> list[dict]:
    final_list = []
    for dim in AVAILABLE_DIMENSIONS:
        # Block Entities
        print(dim, "Block Entities⤵️")
        dim_block_entities = read_drehmal_json_data(dim, "block_entities")
        final_list.extend(extract_data_block_entities(dim_block_entities, dim))
        # Standard Entities
        print(dim, "Standard Entities⤵️")
        dim_entities = read_drehmal_json_data(dim, "entities")
        final_list.extend(extract_data_entities(dim_entities, dim))
    print(f"Total item length from {len(AVAILABLE_DIMENSIONS)} dimensions: {len(final_list)}")
    return final_list

ALL_DATA = get_all_data()

overworld Block Entities⤵️
overworld Standard Entities⤵️
end Block Entities⤵️
end Standard Entities⤵️
lodahr Block Entities⤵️
lodahr Standard Entities⤵️
space Block Entities⤵️
space Standard Entities⤵️
true_end Block Entities⤵️
true_end Standard Entities⤵️
Total item length from 5 dimensions: 13057


In [15]:
SAVE_DIRECTORY = Path("../data_raw/").resolve()

def save_to_json_data(file_name: str, data: list[dict]):
    """Save the data to a JSON file in the SAVE_DIRECTORY.

    :param file_name: file name without .json
    :param data: list of dict data
    """
    save_path = SAVE_DIRECTORY / f"{file_name}.json"
    with open(save_path, 'w') as f:
        json.dump(data, f)

In [54]:
# TEST mini lists and saving first
TEST_FIN_LIST = []
TEST_FIN_LIST.extend(test_be_list)
TEST_FIN_LIST.extend(test_entity_list)
print(len(TEST_FIN_LIST))
save_to_json_data("test_all_entity_data", TEST_FIN_LIST)
print("test saved")

22
test saved


In [16]:
save_to_json_data("all_entity_data", ALL_DATA)
print("done")

done
